# 第4章 优化算法 · 限时测试



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter4/Chapter4.md]]（最速下降、线搜索、Momentum/NAG、矩阵AdaGrad、RMSProp、Adam、AdaMax、小批量）
- [[课程/深度学习/课堂笔记/2026-09-16-第6节-深度学习优化理论讲座.md]]（第四至八节与学习率边界校注）
- [[课程/深度学习/课堂笔记/2026-09-21-第7节-随机优化算法讲解.md]]（第二至五节；Chapter4(1).py程序解读）
- [[课程/深度学习/演示代码/Chapter4(1).py]]（d=5、SGD/Adam/RMSprop、1000步、初始全1）
- [[课程/深度学习/作业/Homework02/Homework02.md]]（Exercise 5、6、8；原参数下如实报告发散）
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]（C4-03—10、A-06、07、10）
- [[课程/深度学习/PPT/2026深度学习第一次上机考试规则与命令表v1.1 (1)/2026深度学习第一次上机考试规则与命令表v1.1 (1).md]]（30分钟、100分及API范围）


## 模拟要求

计时 **30 分钟，100 分**；一道完整编程题、多个小问，**从导入库开始独立编写，不做代码补全**。先作答，再运行自检。题面中的接口名称仅用于统一自检，不提供实现框架。

本地 PNG 写入当前目录 `output/`（请用 pathlib 创建目录）；正式考试若题目指定 `/output/Qxxx.png`，须严格使用正式路径和文件名。练习中的输出目录是本地适配，不是正式考试环境的改动。


## Q4B 三变量耦合系统：用自动微分公平比较 SGD 与 Adam

**30分钟，100分；依据 Chapter4末尾自动微分演示新编，与A卷的手动线性校准不同。**

三变量系统的非线性残差目标为
$F(x)=(x_1x_2-1)^2+(x_2x_3-2)^2+(x_1+x_3-3)^2$。
已知 `(1,1,2)` 可使目标为0，但不要求证明极小点唯一。
初值固定 `(1,1,1)`，CPU float64；比较SGD和Adam，学习率均.05、各400步，不使用外部数据。

1. **20分/约5分钟**：定义 `coupled_objective(x)`，直接由张量运算实现，不把张量转NumPy再求损失。打印初值目标与一次反传梯度，保存 `initial_gradient4b`。
2. **40分/约12分钟**：定义 `optimize_coupled(name)`，每次新建叶参数和优化器，执行清零—前向—反传—更新；返回最终点与401项纯数值日志。保存 `runs4b`（键SGD/Adam）。不能让Adam沿用SGD终点或状态。
3. **20分/约5分钟**：输出各方法终点、最终目标和到 `(1,1,2)` 的距离；重新用终点核对日志末项。用 `initial_gradient4b` 与手算梯度比较，解释没有mini-batch抽样时这里用的是完整目标梯度。
4. **20分/约8分钟**：画两条半对数损失曲线，英文标签，保存 `output/Q4B.png`；比较这一次实验结果，不给出“某优化器永远最好”的结论。

零值对数显示可用 `np.maximum(history,1e-30)`，不得更改真实日志。测试从空白单元独立作答。


In [ ]:
# 从导入库开始，在此独立完成整道编程题。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
import torch
from pathlib import Path
try:
    assert coupled_objective(torch.tensor([1.,1.,1.],dtype=torch.float64)).item()==2 and np.allclose(initial_gradient4b,[-2,-2,-4])
    assert coupled_objective(torch.tensor([1.,1.,2.])).item()==0
    for name,(x,history) in runs4b.items():
        history = np.asarray(history, dtype=float)
        assert history.shape==(401,) and history[0]==2
        assert np.all(np.isfinite(history)) and history[-1]<1e-6
        assert abs(history[-1]-coupled_objective(torch.tensor(x)).item())<1e-12
    assert abs(runs4b['SGD'][1][1]-.9965)<1e-10
    assert not np.allclose(runs4b['SGD'][1][:5],runs4b['Adam'][1][:5])
    assert Path('output/Q4B.png').stat().st_size>1000
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")
